# M2: reviewing the saved baseline results

Mennatalla Mostafa Mohamed Kamal Ibrahem — 2101377

This notebook reviews the earlier M2 run in `results/m2_baselines/`. It checks the saved measurements and plots their mean PSNR. It does not run BM3D or NLM again.

Open it from the repository root or the `notebooks` folder, using Python with Matplotlib installed. The scripts and saved result files must be present. Raw SIDD images and a Kaggle login are not needed for this review.


## 1. Find the project files

The first cell looks for the review script and its M2 evidence file. Checking for both helps avoid opening the wrong project folder or accidentally reading the M1 results.


In [ ]:
from pathlib import Path
import runpy

# Find the project even when this notebook is opened from the notebooks folder.
# Both files must be present so the review uses the intended M2 results.
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'scripts/review_m2_baselines.py').is_file()
                     and (p / 'results/m2_baselines/evidence.json').is_file()), None)
if project_root is None:
    raise RuntimeError('Open the updated project or extracted UPLOAD folder first.')
print('Review root:', project_root)


## 2. Check the data split and saved measurements

The fixed split contains 364 training crops, 120 validation crops and 156 test crops. Captures from the same scene stay together.

The review script checks the file hashes, scene separation, crop counts and crop IDs. It also checks that training rows have no clean-reference paths. That last check applies to the manifest; it does not inspect a separate training program.

PSNR and SSIM averages are recalculated from the per-crop CSV values. Processing time is checked using the median. These checks use the saved measurements, so they do not create a new denoising experiment.


In [ ]:
# Use the same review function as the command-line script.
# It checks the saved files and returns the summaries used in the chart below.
review_module = runpy.run_path(str(project_root / 'scripts/review_m2_baselines.py'))
summaries = review_module['review'](project_root)


## 3. Compare the methods

BM3D uses `sigma_psd = 0.20` and NLM uses `h = 0.08`, selected during M1 calibration. Both are evaluated on the same 120 validation crops.

The bars show mean PSNR from the verified summaries. Higher PSNR means lower pixel-wise error under this evaluation protocol. The printed table also gives SSIM and median processing time, so quality and speed can be considered together.

The chart has no error bars. Variation between individual crops is not the same as uncertainty across independent scenes.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
# Keep the chart order the same as the summaries returned by the review script.
labels = ['Noisy', 'BM3D', 'NLM']
values = [r['psnr_db'] for r in summaries]
bars = ax.bar(labels, values, color=['#94a3b8', '#0072b2', '#d55e00'])

# Round only the displayed labels; keep the saved measurements unchanged.
ax.bar_label(bars, labels=[f'{v:.3f}' for v in values], padding=4)
ax.set(ylim=(0, 43), ylabel='Mean validation PSNR (dB)',
       title='M2 baselines: the same 120 SIDD validation crops')
fig.tight_layout()
plt.show()


## 4. Keep the comparisons separate

| Scope | Crops | BM3D PSNR (dB) | NLM PSNR (dB) |
|---|---:|---:|---:|
| M1 calibration subset | 24 validation | 37.191 | 36.027 |
| M1 held-out test | 156 test | 38.440 | 36.717 |
| Earlier M2 full validation run | 120 validation | 38.148 | 36.527 |

These rows describe different evaluation sets or stages. The 24 calibration crops are included in the 120 validation crops, so the full validation comparison is not an independent held-out test.

This notebook reads the earlier M2 run. The later Colab results are stored separately in `results/mennatalla_m2_2026_11_22/` and checked by `scripts/review_mennatalla_run.py`. Keep each run's quality measurements and timings together.

## 5. Optional: run a new baseline experiment

The command below is for the separately supplied complete `C17_Project` package. It needs `scripts/run.py`, the images in `data/m1` and the project's Python environment. Those components are not all included on this GitHub branch.

From Command Prompt inside that complete project:

```bat
.venv\Scripts\python.exe scripts\run.py evaluate --data data\m1 --method bm3d --split val --output results\menna_bm3d_review_01
```

Use a fresh output folder for each new experiment so previous measurements are preserved. NLM can be run with `--method nlm` and a different output folder. Avoid writing new runs into the saved evidence folders.
